In [8]:
%pip install boto3 pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 27.5 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 3.0.0
    Uninstalling python-dateutil-3.0.0:
      Successfully uninstalled python-dateutil-3.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [boto3]32m4/5 [boto3]sfer]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import json

#configure the environment Before network modules
os.environ["USE_LOCALSTACK"] = "1"
os.environ["LOCALSTACK_URL"] = "http://localhost:4566"

#Ensure state/state.json exists
os.makedirs("state", exist_ok=True)
if not  os.path.exists("state/state.json"):
    with open("state/state.json", "w") as f:
        json.dump({}, f)
print("Environment configured for Floci at :" , os.environ["LOCALSTACK_URL"])

Environment configured for Floci at : http://localhost:4566


In [ ]:

from network.planner import load_config
from network.main import command_plan , command_apply, command_destroy
from network.ec2_instance import create_ec2_instance

config = load_config()
print("Loaded config for vpc:", config["vpc"]["name"])
print(config["security_groups"])

Loaded config for vpc: dev-vpc
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [16]:
plan = command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-547828e0...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-1ead48a2...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-c92c12e3...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-95816d42...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-48bac294...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-687dff55...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-5de57a8ee1afa04b...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-a78530f8...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-b774d2ee...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-07430e2eff1eb9552...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [17]:
command_apply(config)


 DEPLOYING - Building Infra........

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-547828e0...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-1ead48a2...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-c92c12e3...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-95816d42...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-48bac294...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-687dff55...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-5de57a8ee1afa04b...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-a78530f8...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-b774d2ee...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-07430e2eff1eb9552...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.

 Nothing to build. Base is fully operational


In [18]:
instance_id = create_ec2_instance(config)
print("launched EC2 : ", instance_id)

EC2 instance already exsists: i-3a3ccfd98d2ffa625
launched EC2 :  i-3a3ccfd98d2ffa625


In [ ]:
from network.aws_client import get_client
from network.state import load_state

ec2 = get_client('ec2')
state = load_state()

vpcs = ec2.describe_vpcs(VpcIds=[state['vpc_id']])['Vpcs'] if 'vpc_id' in state else []
subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['Subnets'] if 'vpc_id' in state else []
igws = ec2.describe_internet_gateways(InternetGatewayIds=[state['igw_id']])['InternetGateways'] if 'igw_id' in state else []
nats = ec2.describe_nat_gateways(NatGatewayIds=[state['nat_id']])['NatGateways'] if 'nat_id' in state else []
rts = ec2.describe_route_tables(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['RouteTables'] if 'vpc_id' in state else []
sgs = ec2.describe_security_groups(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['SecurityGroups'] if 'vpc_id' in state else []
instances = ec2.describe_instances(InstanceIds=[state['ec2_instance_id']])['Reservations'] if 'ec2_instance_id' in state else []

print(f"VPCs found:              {len(vpcs)}  ({state.get('vpc_id')})")
print(f"Subnets found:           {len(subnets)}  (expected 4)")
print(f"Internet Gateways found: {len(igws)}  ({state.get('igw_id')})")
print(f"NAT Gateways found:      {len(nats)}  ({state.get('nat_id')})")
print(f"Route Tables found:      {len(rts)}  (includes default + 2 custom)")
print(f"Security Groups found:   {len(sgs)}  (includes default + dev-web-sg)")
print(f"EC2 Instances found:     {len(instances)}  ({state.get('ec2_instance_id')})")

VPCs found:              1  (vpc-547828e0)
Subnets found:           4  (expected 4)
Internet Gateways found: 1  (igw-687dff55)
NAT Gateways found:      1  (nat-5de57a8ee1afa04bb)
Route Tables found:      3  (includes default + 2 custom)
Security Groups found:   2  (includes default + dev-web-sg)
EC2 Instances found:     1  (i-3a3ccfd98d2ffa625)


Bad pipe message: %s [b'0.9,image/jxl,image/avif,image/webp,image/apng,*/*;q=0.8\r\nHost: localhost:43383\r\nUser-Agent: Mozilla/5.0 (Window', b'NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/158.0.0.0 Safari/537.36\r\nAccept-Encoding: gzip,']
Bad pipe message: %s [b'eflate, br, zstd\r\nAccept-Langua']
Bad pipe message: %s [b': en-US,en;q=0.7\r\nCache-Control: max-age=0\r\nReferer: https://github.com/\r\nX-Request-ID: 54283c690b449e', b'8801a66857d917c3\r\nX-Real-IP: 167.103.89.28\r\nX-Forwarded']
Bad pipe message: %s [b'ort: 443\r\nX-Forwarded-Scheme: https\r\nX-Origi']
Bad pipe message: %s [b'l-URI: /\r\nX-Scheme: https\r\nsec-gpc: 1\r\nsec-fetch-site: cross-site\r\nsec-fetch-mode: navigate\r\nsec-fetch-user: ', b'\r\nsec-fetch-dest: document\r\nsec-ch-ua: "Chromium";v="158", "No', b'A/Brand";v="24", "Brave";v="158"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"\r\npriority: u=0, i\r\nX-Forwarde']
Bad pipe message: %s [b'Proto: https\r\nX-Forwarded-Host:

In [ ]:
import json
from network.aws_client import get_client
from network.state import load_state, save_state 
iam = get_client("iam")
ecs = get_client("ecs")
state = load_state()

# Cretae a Task Execution Role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {}
    }]
}
